In [ ]:
import glob
import os 
import numpy as np
from tqdm import tqdm
import skimage

In [ ]:
root_folder = "./data/expanded_ss316l_all_laser_velocity_xz_cross_section_data_expanded_frame"
old_prefix = "expanded_frame"
new_prefix = "expanded_frame_wih_fluid_fraction"
new_root_folder = root_folder.replace(old_prefix, new_prefix)
os.makedirs(new_root_folder, exist_ok=True)

In [ ]:
all_folders = glob.glob(os.path.join( "./data/expanded_ss316l_all_laser_velocity_xz_cross_section_data_expanded_frame", '**/*npy'), recursive = True)

In [ ]:
# np.load(all_folders[0]).shape
for file in tqdm(all_folders):
    assert len(np.load(file).shape) == 2

In [ ]:
from scipy.ndimage.morphology import distance_transform_bf

import matplotlib.pyplot as plt
def fill_keyhole(temp, borderline = 293, plate_height = 30, plot = False):
    '''
    Temp: Three dimensional temperature array
    '''
    temp = np.array(temp,dtype = float)[:, :plate_height]

    labeled_image = skimage.measure.label(temp>borderline, background = 1)
    # plt.imshow(labeled_image)
    # plt.title('label')
    # plt.show()
    props = skimage.measure.regionprops(labeled_image)
    volume = [prop.area for prop in props]
    
    idx_background = np.argmax(volume)
    for index, prop in enumerate(props):

        bbox = prop.bbox
        if bbox[-1] < temp.shape[-1]:
            continue
        if index == idx_background:
            continue
        else:
            temp[labeled_image == prop.label] = 10000
    if plot:
        plt.figure(dpi = 300, figsize = np.array([4,3])*1.15)
        
        plt.imshow(temp.T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000)
        plt.title("Melt Pool, Keyhole Filled")
        plt.axis('off')
        plt.show()
        # plt.imshow(temp.T, origin  = 'lower')
        # plt.show()
    return temp
def extract_keyhole_temperatures(sample, plate_height, plot= False):

    if plot:
        plt.figure(dpi = 300, figsize = np.array([4,3])*1.15)
        
        plt.imshow(sample[:, :plate_height].T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000)
        plt.title("Melt Pool")
        plt.axis('off')
        plt.show()
    temp_2d = fill_keyhole(sample, borderline = 1000, plate_height = 30)
    if plot:
        plt.imshow(temp_2d.T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000)
        plt.title('filled')
        plt.show()
    criterion = np.logical_and(distance_transform_bf(temp_2d<5000)<1.1 , distance_transform_bf(temp_2d<5000)>0)
    filtered_temperatures = criterion*temp_2d
    temperature_values = filtered_temperatures[filtered_temperatures>0]
    if plot:
        plt.figure(dpi = 300, figsize = np.array([4,3])*1.15)
        plt.axis('off')
        plt.title('Keyhole Boundary Temperature Values')
        plt.imshow((criterion*temp_2d).T, origin = 'lower', vmin = 293, vmax = 5000, cmap ='jet')
        plt.show()
    return temperature_values
def get_fluid_fraction(temperature, plate_height = 30):
    temp_2d = fill_keyhole(temperature, borderline = 1000, plate_height = plate_height)
    above_plate = temperature[:, plate_height:]
    criterion_2 = above_plate > 300
    criterion_1 = temp_2d<8000
    fluid_fraction = np.hstack((criterion_1, criterion_2))
    return fluid_fraction

In [ ]:

# plt.imshow(get_fluid_fraction(gts[3][0][0], plate_height = 30).T, origin = 'lower')
# plt.colorbar()  

In [ ]:
# _, filtered_contour, pores = find_keyhole_boundary(temp_2d, threshold = 1700, plate_height = 30)
for i in range(10):
    sample_index = np.random.randint(0, len(all_folders)) 
    temperature_sample = np.load(all_folders[sample_index])
    if np.shape(temperature_sample)[-1] == 30:
        # LR sample
        plate_height = 20
    elif np.shape(temperature_sample)[-1] == 60:
        # HR sample
        plate_height = 40
    else:
        print("Shape not recognized")
    plt.imshow(temperature_sample.T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000)
    plt.axhline(plate_height)
    plt.show()
    fluid_fraction = get_fluid_fraction(temperature_sample, plate_height = plate_height)
    # plt.imshow(get_fluid_fraction(np.load(all_folders[0]), plate_height = 30))
    plt.imshow(fluid_fraction.T, origin = 'lower')
    plt.show()
    plt.imshow(temperature_sample.T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000, alpha = 0.5)
    plt.imshow(fluid_fraction.T, origin = 'lower', alpha = 0.5, cmap = 'gray')
    plt.show()

In [ ]:
np.dstack([temperature_sample, fluid_fraction])

In [ ]:
all_folders[0].replace('expanded_ss316l_all_laser_velocity_xz_cross_section_data_expanded_frame', 'expanded_ss316l_all_laser_velocity_xz_cross_section_data_expanded_frame_fluid_fraction')

In [ ]:
all_folders_sorted = sorted(all_folders)
for i, file in enumerate(tqdm(all_folders_sorted)):
    temperature_sample = np.load(file)
    if np.shape(temperature_sample)[-1] == 30:
        # LR sample
        plate_height = 20
    elif np.shape(temperature_sample)[-1] == 60:
        # HR sample
        plate_height = 40
    else:
        print("Shape not recognized")
    fluid_fraction = get_fluid_fraction(temperature_sample, plate_height = plate_height)
    fluid_fraction = np.expand_dims(fluid_fraction, axis = 0)
    temperature_sample = np.expand_dims(temperature_sample, axis = 0)
    new_file = file.replace('expanded_ss316l_all_laser_velocity_xz_cross_section_data_expanded_frame', 'expanded_ss316l_all_laser_velocity_xz_cross_section_data_expanded_frame_fluid_fraction')
    if not os.path.exists(os.path.dirname(new_file)):
        os.makedirs(os.path.dirname(new_file))
    np.save(new_file, np.dstack([temperature_sample[0], fluid_fraction[0]]))
    # print(new_file)
    if i % 5000 == 0:
        print(new_file)
    if i % 10000 == 0:
        plt.imshow(temperature_sample[0].T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000)
        plt.axhline(plate_height)
        plt.show()
        plt.imshow(fluid_fraction[0].T, origin = 'lower')
        plt.show()
        plt.imshow(temperature_sample[0].T, origin = 'lower', cmap = 'jet', vmin = 293, vmax = 5000, alpha = 0.5)
        plt.imshow(fluid_fraction[0].T, origin = 'lower', alpha = 0.5, cmap = 'gray')
        plt.show()

In [ ]:
np.dstack([temperature_sample[0], fluid_fraction[0]])[:,:, None].shape

In [ ]:
fluid_fraction.shape